# Weekly Project 09 - Visual Odometry

Today you are going to implement the last parts of the algorithm you started on monday. For reference you can see it below.

![title](algorithm_3.png)

It is a good idea to follow and track the steps in the algorithm in the below implementation. Only take one step at a time.

Once you have the algorithm up and running you can try with a larger dataset to see if your algorithm is able to maintain good accurracy over a longer distance. The larger dataset can be found here:
[Left images](https://dtudk-my.sharepoint.com/:u:/g/personal/evanb_dtu_dk/EQu8kmGBDDROtGJ7IkZB2tQBJrxmgY9t8LVM_JuEi83TYw)
[Right images](https://dtudk-my.sharepoint.com/:u:/g/personal/evanb_dtu_dk/EcKI_zrXTvpMulizidCZm4oBLJcQ_LTV9Zs6oQFF74JTRQ)

## Weekly Project 09 - Start Here

In [31]:
import numpy as np
import cv2 as cv2
import glob

def getK():
    return np.array([[7.188560e+02, 0.000000e+00, 6.071928e+02],
                     [0, 7.188560e+02, 1.852157e+02],
                     [0, 0, 1]])

def getTruePose():
    file = 'data/00.txt'
    return np.genfromtxt(file, delimiter=' ', dtype=None)

<details>
  <summary>tips:</summary>

- [Feature Matching](https://docs.opencv.org/4.x/dc/dc3/tutorial_py_matcher.html)
- To get the keypoint of a certain match do: ```kp1[match.queryIdx].pt``` and ```kp2[match.trainIdx].pt```

To complete this function, we followed the “Brute-Force Matching with SIFT Descriptors and Ratio Test” section of the OpenCV documentation. SIFT detects keypoints and computes descriptors in both images, and knnMatch finds the two closest matches for each descriptor. Lowe’s ratio test retains the best match when its distance is less than 75% of the second-best distance. We store the corresponding 2D coordinates in both images, then triangulate them using the camera calibration and stereo baseline. Finally, we convert the homogeneous coordinates to 3D coordinates and return them alongside the matched points in the left image.

In [32]:
def extract_keypoints_sift(img1, img2, K, baseline):
    """
    Use SIFT to detect keypoints and compute descriptors in both images,
    and find matches between them using knnMatch with k=2
    """
    # sift = ...
    sift = cv2.SIFT_create()
    kp1, des1 = sift.detectAndCompute(img1, None)
    kp2, des2 = sift.detectAndCompute(img2, None)

    bf = cv2.BFMatcher()
    matches = bf.knnMatch(des1, des2, k=2)

    """
    Get the best keypoints by applying the Lowes ratio test
    """
    # match_points1 = []
    # ...
    match_points1 = []
    match_points2 = []
    for m, n in matches:
        if m.distance < 0.75 * n.distance:
            match_points1.append(kp1[m.queryIdx].pt)
            match_points2.append(kp2[m.trainIdx].pt)
    
    # Transform best keypoints to nparray
    p1 = np.array(match_points1).astype('float32')
    p2 = np.array(match_points2).astype('float32')

    ##### ############# ##########
    ##### Do Triangulation #######
    ##### ########################
    # project the feature points to 3D with triangulation
    # projection matrix for Left and Right Image
    M_left = K.dot(np.hstack((np.eye(3), np.zeros((3, 1)))))
    M_rght = K.dot(np.hstack((np.eye(3), np.array([[-baseline, 0, 0]]).T)))

    p1_flip = np.vstack((p1.T, np.ones((1, p1.shape[0]))))
    p2_flip = np.vstack((p2.T, np.ones((1, p2.shape[0]))))

    P = cv2.triangulatePoints(M_left, M_rght, p1_flip[:2], p2_flip[:2])

    # Normalize homogeneous coordinates (P->Nx4  [N,4] is the normalizer/scale)
    P = P / P[3]
    land_points = P[:3]

    return land_points.T, p1

This function was already implemented in exercise 1. This function uses pyramidal Lucas–Kanade optical flow to track points from the previous image to the next. It keeps only points successfully tracked (status == 1) and applies the same filter to the previous 2D points, tracked 2D points, and corresponding 3D world points, preserving their correspondence. It returns these three filtered arrays.

In [33]:
def featureTracking(prev_img, next_img, prev_points, world_points):
    """
    Use OpenCV to find the prev_points from the prev_img in the next_img
    Remember to remove points that could not be found from prev_points, next_points, and world_points
    hint: status == 1
    """
    params = dict(winSize=(21, 21),
                 maxLevel=3,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
    
    #next_points, status, _ = ...
    next_points, status, _ = cv2.calcOpticalFlowPyrLK(prev_img, next_img, prev_points, None, **params)
    # keep only points that were successfully tracked (status == 1)
    status = status.reshape(-1).astype(bool)
    world_points = np.asarray(world_points)[status]
    prev_points = prev_points[status]
    next_points = next_points[status]   

    return world_points, prev_points, next_points


### Main loop – `playImageSequence`

The function implements Algorithm 3 using the stereo pair, so the known baseline (0.54 m) gives metric scale.

- **1.2–1.3:** Features from the first stereo pair are matched and triangulated. The first camera pose defines the world frame.
- **2.2:** The previous 2D points are tracked into the current left image with Lucas–Kanade optical flow, following Exercise 1b.
- **2.3:** The camera pose is estimated with `solvePnPRansac` from the 3D-to-2D correspondences, following Exercise 1b. The camera position in the world is $C = -R^T t$.
- **2.4:** New landmarks are triangulated from the current stereo pair and transformed to the world frame with $[R^T | C]$, so that the next PnP returns an absolute pose.

In [4]:
def playImageSequence(l_img_paths, r_img_paths, K):
    baseline = 0.54
    left_img = cv2.imread(l_img_paths[0], 0)
    right_img = cv2.imread(r_img_paths[0], 0)

    ##### ################################# #######
    ##### Get 3D points Using Triangulation #######
    ##### #########################################
    """
    Implement step 1.2 and 1.3
    Store the features in 'reference_2D' and the 3D points (landmarks) in 'landmark_3D'
    hint: use 'extract_keypoints_sift' above
    """
    landmark_3D, reference_2D = extract_keypoints_sift(left_img, right_img, K, baseline)

    # reference
    reference_img = left_img

    # Groundtruth for plot
    truePose = getTruePose()
    traj = np.zeros((600, 600, 3), dtype=np.uint8)
    maxError = 0

    for i in range(1, 101):
        print('image: ', i)
        curImage = cv2.imread(l_img_paths[i], 0)
        curImage_R = cv2.imread(r_img_paths[i], 0)

        ##### ############################################################# #######
        ##### Calculate 2D and 3D feature correspndances in t=T-1, and t=T  #######
        ##### #####################################################################
        """
        Implement step 2.2)
        Remember this is a part of a loop, so the initial features are already
        provided in step 1)-1.3) outside the loop in 'reference_2D' and 'landmark_3D'
        """
        landmark_3D, reference_2D, tracked_2Dpoints = featureTracking(reference_img, curImage, reference_2D, landmark_3D)


        ##### ####################################### #######
        ##### Calculate relative pose using PNPRansac #######
        ##### ###############################################
        """
        Implement step 2.3)
        """
        _, rvec, tvec, inliers = cv2.solvePnPRansac(
                        landmark_3D.astype(np.float64),
                        tracked_2Dpoints.astype(np.float64),
                        K.astype(np.float64),
                        distCoeffs=None,
                    )
        

        ##### ####################################################### #######
        ##### Get Pose and Tranformation Matrix in world coordionates #######
        ##### ###############################################################
        rot, _ = cv2.Rodrigues(rvec)
        tvec = -rot.T.dot(tvec)  # coordinate transformation, from camera to world. What is the XYZ of the camera wrt World
        inv_transform = np.hstack((rot.T, tvec))  # inverse transform. A tranform projecting points from the camera frame to the world frame

        ##### ################################# #######
        ##### Get 3D points Using Triangulation #######
        ##### #########################################
        # re-obtain the 3D points. 
        # hint: use 'extract_keypoints_sift' again
        """
        Implement step 2.4)
        """
        landmark_3D_new, reference_2D_new = extract_keypoints_sift(curImage, curImage_R, K, baseline)
        
        
        #Project the points from camera to world coordinates
        reference_2D = reference_2D_new.astype('float32')
        landmark_3D = inv_transform.dot(np.vstack((landmark_3D_new.T, np.ones((1, landmark_3D_new.shape[0])))))
        landmark_3D = landmark_3D.T

        ##### ####################### #######
        ##### Done, Next image please #######
        ##### ###############################
        reference_img = curImage

        ##### ################################## #######
        ##### START OF Print and visualize stuff #######
        ##### ##########################################
        # draw images
        draw_x, draw_y = int(tvec[0]) + 300, 600-(int(tvec[2]) + 100)
        true_x, true_y = int(truePose[i][3]) + 300, 600-(int(truePose[i][11]) + 100)

        curError = np.sqrt(
            (tvec[0] - truePose[i][3]) ** 2 +
            (tvec[1] - truePose[i][7]) ** 2 +
            (tvec[2] - truePose[i][11]) ** 2)
        
        if (curError > maxError):
            maxError = curError

        print(tvec[0],tvec[1],tvec[2], rvec[0], rvec[1], rvec[2])
        print([truePose[i][3], truePose[i][7], truePose[i][11]])
        
        text = "Coordinates: x ={0:02f}m y = {1:02f}m z = {2:02f}m".format(float(tvec[0]), float(tvec[1]),
                                                                           float(tvec[2]))
        cv2.circle(traj, (draw_x, draw_y), 1, (0, 0, 255), 2)
        cv2.circle(traj, (true_x, true_y), 1, (255, 0, 0), 2)
        cv2.rectangle(traj, (10, 30), (550, 50), (0, 0, 0), cv2.FILLED)
        cv2.putText(traj, text, (10, 50), cv2.FONT_HERSHEY_PLAIN, 1, (255, 255, 255), 1, 8)

        h1, w1 = traj.shape[:2]
        h2, w2 = curImage.shape[:2]
        vis = np.zeros((max(h1, h2), w1 + w2, 3), np.uint8)
        vis[:h1, :w1, :3] = traj
        vis[:h2, w1:w1 + w2, :3] = np.dstack((np.dstack((curImage,curImage)),curImage))

        cv2.imshow("Trajectory", vis)
        k = cv2.waitKey(1) & 0xFF
        if k == 27: break


    cv2.waitKey(0)
    cv2.destroyAllWindows()
    print('Maximum Error: ', maxError)
    ##### ################################ #######
    ##### END OF Print and visualize stuff #######
    ##### ########################################

In [ ]:
# Load image paths
left_img_paths = sorted(glob.glob('data/left/*.png'))
right_img_paths = sorted(glob.glob('data/right/*.png'))

K = getK()

playImageSequence(left_img_paths, right_img_paths, K)

image:  1


/var/folders/wh/6d7zf7cx3632z8j44tx9fqxr0000gn/T/ipykernel_65781/3805584563.py:86: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  draw_x, draw_y = int(tvec[0]) + 300, 600-(int(tvec[2]) + 100)
/var/folders/wh/6d7zf7cx3632z8j44tx9fqxr0000gn/T/ipykernel_65781/3805584563.py:100: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  text = "Coordinates: x ={0:02f}m y = {1:02f}m z = {2:02f}m".format(float(tvec[0]), float(tvec[1]),
/var/folders/wh/6d7zf7cx3632z8j44tx9fqxr0000gn/T/ipykernel_65781/3805584563.py:101: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single elem

[-0.00227842] [-0.00703923] [0.67547519] [-0.00217427] [0.00334328] [-0.00261036]
[-0.04690294, -0.02839928, 0.8586941]
image:  2
[-0.01421229] [-0.01295029] [1.37229754] [-0.00388047] [0.00737929] [-0.00082554]
[-0.09374345, -0.05676064, 1.716275]
image:  3
[-0.03299899] [-0.01979135] [2.08589033] [-0.00542574] [0.0111697] [-0.00062243]
[-0.1406429, -0.08515762, 2.574964]
image:  4
[-0.05167471] [-0.0335308] [2.81642371] [-0.00589889] [0.0160554] [0.00109477]
[-0.1874858, -0.1135202, 3.432648]
image:  5
[-0.06795917] [-0.04547951] [3.55923965] [-0.0060917] [0.02061187] [7.70699396e-05]
[-0.2343818, -0.141915, 4.291335]
image:  6
[-0.0934416] [-0.05798517] [4.31460822] [-0.00739334] [0.02516383] [0.00216643]
[-0.2812195, -0.1702743, 5.148987]
image:  7
[-0.12680478] [-0.07018718] [5.08676188] [-0.00802003] [0.02996213] [0.00496733]
[-0.3281178, -0.1986703, 6.007777]
image:  8
[-0.1609621] [-0.07956575] [5.87431325] [-0.00656749] [0.03427284] [0.00543799]
[-0.3749547, -0.227029, 6.86547

### Challenge
The current implementation only uses features computed at the current timestep. However, as we process more images we potentially have a lot of features from previous timesteps that are still valid. The challenge is to expand the `extract_keypoints_surf(..., refPoints)` function by giving it old reference points. You should then combine your freshly computed features with the old features and remove all duplicates. This requires you to keep track of old features and 3D points.

Hint 1: The following function `removeDuplicate` can be used for removing duplicates.

Hint 2: you are not interested in points that are behind you, so remember to remove points that are negative in the direction you move.

### Challenge – reusing old features

Instead of discarding all features after each frame, previously tracked points are kept between iterations:

- The tracked points that PnP accepted as inliers are retained together with their world-frame landmarks.
- Old landmarks that are now behind the camera (negative depth in the camera frame) are removed.
- `extract_keypoints_sift` is extended with a `refPoints` argument. New SIFT matches lying within 5 px of an existing point are discarded with `removeDuplicate`.
- The remaining new points are triangulated, transformed to the world frame and combined with the old ones.

The number of frames is taken from the image list, so the same function can be run on the larger dataset.

In [36]:
def removeDuplicate(queryPoints, refPoints, radius=5):
    # remove duplicate points from new query points,
    for i in range(len(queryPoints)):
        query = queryPoints[i]
        xliml, xlimh = query[0] - radius, query[0] + radius
        yliml, ylimh = query[1] - radius, query[1] + radius
        inside_x_lim_mask = (refPoints[:, 0] > xliml) & (refPoints[:, 0] < xlimh)
        curr_kps_in_x_lim = refPoints[inside_x_lim_mask]

        if curr_kps_in_x_lim.shape[0] != 0:
            inside_y_lim_mask = (curr_kps_in_x_lim[:, 1] > yliml) & (curr_kps_in_x_lim[:, 1] < ylimh)
            curr_kps_in_x_lim_and_y_lim = curr_kps_in_x_lim[inside_y_lim_mask, :]
            if curr_kps_in_x_lim_and_y_lim.shape[0] != 0:
                queryPoints[i] = np.array([0, 0])
    return (queryPoints[:, 0] != 0)

def extract_keypoints_sift(img1, img2, K, baseline, refPoints=None):
    sift = cv2.SIFT_create()
    kp1, des1 = sift.detectAndCompute(img1, None)
    kp2, des2 = sift.detectAndCompute(img2, None)

    bf = cv2.BFMatcher()
    matches = bf.knnMatch(des1, des2, k=2)

    match_points1 = []
    match_points2 = []
    for m, n in matches:
        if m.distance < 0.75 * n.distance:
            match_points1.append(kp1[m.queryIdx].pt)
            match_points2.append(kp2[m.trainIdx].pt)

    p1 = np.array(match_points1).astype('float32')
    p2 = np.array(match_points2).astype('float32')

    # drop new matches that lie within 5 px of a point we are already tracking
    if refPoints is not None:
        mask = removeDuplicate(p1.copy(), refPoints)
        p1 = p1[mask]
        p2 = p2[mask]

    M_left = K.dot(np.hstack((np.eye(3), np.zeros((3, 1)))))
    M_rght = K.dot(np.hstack((np.eye(3), np.array([[-baseline, 0, 0]]).T)))

    p1_flip = np.vstack((p1.T, np.ones((1, p1.shape[0]))))
    p2_flip = np.vstack((p2.T, np.ones((1, p2.shape[0]))))

    P = cv2.triangulatePoints(M_left, M_rght, p1_flip[:2], p2_flip[:2])
    P = P / P[3]
    land_points = P[:3]

    return land_points.T, p1

def playImageSequence_challenge(l_img_paths, r_img_paths, K,n_frames = None):
    baseline = 0.54
    left_img = cv2.imread(l_img_paths[0], 0)
    right_img = cv2.imread(r_img_paths[0], 0)

    landmark_3D, reference_2D = extract_keypoints_sift(left_img, right_img, K, baseline)
    reference_img = left_img

    truePose = getTruePose()
    traj = np.zeros((600, 600, 3), dtype=np.uint8)
    maxError = 0
    
    if n_frames is None:
        n_frames = len(l_img_paths)


    for i in range(1, n_frames):
        print('image: ', i)
        curImage = cv2.imread(l_img_paths[i], 0)
        curImage_R = cv2.imread(r_img_paths[i], 0)

        # 2.2
        landmark_3D, reference_2D, tracked_2Dpoints = featureTracking(reference_img, curImage, reference_2D, landmark_3D)

        # 2.3
        _, rvec, tvec, inliers = cv2.solvePnPRansac(landmark_3D.astype(np.float64),
                                                    tracked_2Dpoints.astype(np.float64),
                                                    K.astype(np.float64), None)

        rot, _ = cv2.Rodrigues(rvec)
        tvec = -rot.T.dot(tvec)
        inv_transform = np.hstack((rot.T, tvec))

        # keep the old points that PnP accepted as inliers
        old_2D = tracked_2Dpoints[inliers.ravel()]
        old_3D = landmark_3D[inliers.ravel()]

        # remove old landmarks that are now behind the camera (negative depth in camera frame)
        z_cam = rot.dot(old_3D.T - tvec)[2]
        old_2D = old_2D[z_cam > 0]
        old_3D = old_3D[z_cam > 0]

        # 2.4 – only new points that are not duplicates of old ones
        landmark_3D_new, reference_2D_new = extract_keypoints_sift(curImage, curImage_R, K, baseline, old_2D)
        landmark_3D_new = inv_transform.dot(np.vstack((landmark_3D_new.T, np.ones((1, landmark_3D_new.shape[0]))))).T

        # combine old and new points
        reference_2D = np.vstack((old_2D, reference_2D_new)).astype('float32')
        landmark_3D = np.vstack((old_3D, landmark_3D_new))

        reference_img = curImage

        draw_x, draw_y = int(tvec[0]) + 300, 600-(int(tvec[2]) + 100)
        true_x, true_y = int(truePose[i][3]) + 300, 600-(int(truePose[i][11]) + 100)

        curError = np.sqrt(
            (tvec[0] - truePose[i][3]) ** 2 +
            (tvec[1] - truePose[i][7]) ** 2 +
            (tvec[2] - truePose[i][11]) ** 2)
        if (curError > maxError):
            maxError = curError

        print(tvec[0], tvec[1], tvec[2], rvec[0], rvec[1], rvec[2])
        print([truePose[i][3], truePose[i][7], truePose[i][11]])

        text = "Coordinates: x ={0:02f}m y = {1:02f}m z = {2:02f}m".format(float(tvec[0]), float(tvec[1]), float(tvec[2]))
        cv2.circle(traj, (draw_x, draw_y), 1, (0, 0, 255), 2)
        cv2.circle(traj, (true_x, true_y), 1, (255, 0, 0), 2)
        cv2.rectangle(traj, (10, 30), (550, 50), (0, 0, 0), cv2.FILLED)
        cv2.putText(traj, text, (10, 50), cv2.FONT_HERSHEY_PLAIN, 1, (255, 255, 255), 1, 8)

        h1, w1 = traj.shape[:2]
        h2, w2 = curImage.shape[:2]
        vis = np.zeros((max(h1, h2), w1 + w2, 3), np.uint8)
        vis[:h1, :w1, :3] = traj
        vis[:h2, w1:w1 + w2, :3] = np.dstack((np.dstack((curImage, curImage)), curImage))

        cv2.imshow("Trajectory", vis)
        k = cv2.waitKey(1) & 0xFF
        if k == 27: break

    cv2.waitKey(0)
    cv2.destroyAllWindows()
    print('Maximum Error: ', maxError)

In [37]:
challenge_left_paths = sorted(glob.glob(r'C:\Users\sdouz\Desktop\DTU\Semester 1\34759 - Perception for Autonomous systems\big_data\bigleft\*.png'))
challenge_right_paths = sorted(glob.glob(r'C:\Users\sdouz\Desktop\DTU\Semester 1\34759 - Perception for Autonomous systems\big_data\bigright\*.png'))

challenge_K = np.array([
    [7.188560e+02, 0.0, 6.071928e+02],
    [0.0, 7.188560e+02, 1.852157e+02],
    [0.0, 0.0, 1.0]
], dtype=np.float64)

playImageSequence_challenge(challenge_left_paths,challenge_right_paths,challenge_K)

image:  1
[-0.00227636] [-0.00703882] [0.67547862] [-0.0021743] [0.0033434] [-0.00261038]
[np.float64(-0.04690294), np.float64(-0.02839928), np.float64(0.8586941)]
image:  2


C:\Users\sdouz\AppData\Local\Temp\ipykernel_14760\2930908164.py:105: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  draw_x, draw_y = int(tvec[0]) + 300, 600-(int(tvec[2]) + 100)
C:\Users\sdouz\AppData\Local\Temp\ipykernel_14760\2930908164.py:118: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  text = "Coordinates: x ={0:02f}m y = {1:02f}m z = {2:02f}m".format(float(tvec[0]), float(tvec[1]), float(tvec[2]))


[-0.01204503] [-0.0121154] [1.37666005] [-0.0037561] [0.00759281] [-0.00109663]
[np.float64(-0.09374345), np.float64(-0.05676064), np.float64(1.716275)]
image:  3
[-0.03161761] [-0.01811949] [2.08734805] [-0.00519486] [0.01145247] [-0.00100673]
[np.float64(-0.1406429), np.float64(-0.08515762), np.float64(2.574964)]
image:  4
[-0.05079165] [-0.02464766] [2.82419141] [-0.00586086] [0.01640811] [0.00032819]
[np.float64(-0.1874858), np.float64(-0.1135202), np.float64(3.432648)]
image:  5
[-0.06664903] [-0.03611983] [3.55753321] [-0.00598185] [0.02088152] [-0.00075521]
[np.float64(-0.2343818), np.float64(-0.141915), np.float64(4.291335)]
image:  6
[-0.0914341] [-0.04853779] [4.30886744] [-0.00711075] [0.02551211] [0.00116762]
[np.float64(-0.2812195), np.float64(-0.1702743), np.float64(5.148987)]
image:  7
[-0.12561617] [-0.05530558] [5.09253193] [-0.00790714] [0.03060402] [0.00381859]
[np.float64(-0.3281178), np.float64(-0.1986703), np.float64(6.007777)]
image:  8
[-0.1633692] [-0.06824376]

## Declaration of AI Usage

- We used AI to help us refine the code documentation.
- We used AI to help us with the challange.